# 🎛️ YuE2 Studio - Pełne Środowisko GPU (Tesla T4 / L4 / A100) w Google Colab
Oficjalna instalacja **YuE2 (multimodal-art-projection)** z nakładką **YuE2 Studio**, obsługą **GGUF (audio.cpp)**, lokalnym asystentem **AI Music Producer (Qwen 2.5 3B)** oraz tunelem Cloudflare (dostęp na telefon i PC).

> ⚠️ **KROK 0 (Włącz GPU):**
> W menu u góry kliknij: **Runtime (Środowisko wykonawcze)** ➔ **Change runtime type (Zmień typ środowiska)** ➔ wybierz **T4 GPU** (lub L4/A100) ➔ **Save (Zapisz)**.

## 🚀 Krok 1: Instalacja bazy YuE2 + Nakładki YuE2 Studio + Silnika GGUF (audio.cpp)

In [ ]:
#@title 1. Pełna instalacja YuE2 + YuE2 Studio + audio.cpp (GGUF) ⚙️
import os
import sys

# 1. Sprawdzenie GPU
try:
    import torch
    cuda_available = torch.cuda.is_available()
except ImportError:
    cuda_available = False

if not cuda_available:
    print("⚠️ UWAGA: Google Colab działa w trybie CPU! Włącz T4 GPU w Runtime -> Change runtime type.\n")

# Ustawienie zmiennych CUDA
if os.path.exists('/usr/local/cuda'):
    os.environ['CUDA_HOME'] = '/usr/local/cuda'
    os.environ['PATH'] = f"/usr/local/cuda/bin:{os.environ.get('PATH', '')}"
    os.environ['LD_LIBRARY_PATH'] = f"/usr/local/cuda/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"

# Wyłączenie konfliktowej biblioteki torchvision dla modeli audio
os.environ['TRANSFORMERS_NO_TORCHVISION'] = '1'
!pip uninstall -y -q torchvision

print("📦 1. Instalacja zależności PyTorch, Transformers & Accelerate...")
!pip install -q --upgrade pip setuptools wheel
!pip install -q --upgrade "transformers[torch]>=4.40.0" accelerate safetensors einops soundfile>=0.12.1 numpy>=1.24.0 tiktoken>=0.7.0 huggingface_hub>=0.23.0 scipy>=1.11.0 cmake ninja peft

print("📦 2. Klonowanie oficjalnego repozytorium YuE2...")
if not os.path.exists('/content/YuE'):
    !git clone https://github.com/multimodal-art-projection/YuE.git /content/YuE
    %cd /content/YuE
    !git checkout 92a73cc7652fcc1f937855e4b765e0a0edd7ff2e

%cd /content/YuE

print("📦 3. Klonowanie i instalacja nakładki YuE2 Studio...")
if not os.path.exists('/content/YuE/Yue2_Studio'):
    !git clone https://github.com/vrgamegirl19/Yue2_Studio.git /content/YuE/Yue2_Studio

# Instalacja nakładki Studio do folderu YuE2
!python /content/YuE/Yue2_Studio/install_studio.py --target /content/YuE

# Patch dla kart Tesla T4 (eliminacja 'No available kernel' poprzez bezpieczny fallback SDPA)
cuda_graph_py = '/content/YuE/src/yue2/cuda_graph.py'
if os.path.exists(cuda_graph_py):
    with open(cuda_graph_py, 'r', encoding='utf-8') as f:
        content = f.read()
    content = content.replace(
        'flash = fused and torch.backends.cuda.is_flash_attention_available()',
        'cap = torch.cuda.get_device_capability(self.device) if self.device.type == "cuda" else (0, 0)\n        flash = fused and cap[0] >= 8 and torch.backends.cuda.is_flash_attention_available()'
    )
    content = content.replace(
        'attention_backend = "flash" if flash else "cudnn" if fused and torch.backends.cudnn.is_available() else "sdpa"',
        'attention_backend = "flash" if flash else "sdpa"'
    )
    content = content.replace(
        'elif self.attention_backend == "cudnn":\n                from torch.nn.attention import SDPBackend, sdpa_kernel\n                with sdpa_kernel(SDPBackend.CUDNN_ATTENTION):\n                    h = F.scaled_dot_product_attention(q.transpose(1, 2), keys.transpose(1, 2), values.transpose(1, 2),\n                            attn_mask=visible, is_causal=False,\n                            enable_gqa=config.num_attention_heads != config.num_key_value_heads).transpose(1, 2)',
        ''
    )
    with open(cuda_graph_py, 'w', encoding='utf-8') as f:
        f.write(content)

# Patch automatycznego wykrywania ścieżki audiocpp_cli w gguf.py
gguf_py = '/content/YuE/src/yue2_studio/gguf.py'
if os.path.exists(gguf_py):
    with open(gguf_py, 'r', encoding='utf-8') as f:
        gcontent = f.read()
    gcontent = gcontent.replace(
        "cfg=settings['gguf'];exe=Path(cfg['executable'])",
        "cfg=settings['gguf']\n    exe_raw = cfg.get('executable', '').strip()\n    exe = Path(exe_raw) if exe_raw else Path('')\n    if not exe.is_file():\n        for cand in ('/content/YuE/audiocpp_cli', '/usr/local/bin/audiocpp_cli', '/content/YuE/tools/audio.cpp/build/bin/audiocpp_cli'):\n            if os.path.isfile(cand):\n                exe = Path(cand); cfg['executable'] = cand; break"
    )
    with open(gguf_py, 'w', encoding='utf-8') as f:
        f.write(gcontent)

# Sprawdzenie czy gotowy silnik audiocpp_cli istnieje (np. na Google Drive lub lokalnie)
has_prebuilt = False
for prebuilt_cand in ('/content/drive/MyDrive/audiocpp_cli', '/content/audiocpp_cli', '/content/YuE/audiocpp_cli'):
    if os.path.isfile(prebuilt_cand):
        !cp {prebuilt_cand} /content/YuE/audiocpp_cli
        !cp {prebuilt_cand} /usr/local/bin/audiocpp_cli
        !chmod +x /content/YuE/audiocpp_cli /usr/local/bin/audiocpp_cli
        print(f"⚡ Znaleziono gotowy plik audiocpp_cli ({prebuilt_cand})! Kopiowanie (1 sekunda, bez kompilacji)...")
        has_prebuilt = True
        break

if not has_prebuilt:
    # Kompilacja silnika GGUF audio.cpp pod Linux/CUDA
    print("⚡ 4. Klonowanie i budowanie silnika GGUF (audio.cpp CUDA sm_75+)...")
    !git config --global url."https://github.com/".insteadOf "git@github.com:"
    if not os.path.exists("/content/YuE/tools/audio.cpp"):
        !git clone --branch dev https://github.com/0xShug0/audio.cpp.git /content/YuE/tools/audio.cpp
        %cd /content/YuE/tools/audio.cpp
        !git checkout fbe3eedbf6c504e45189e2cdcf1b257740a28863

    %cd /content/YuE/tools/audio.cpp
    !bash scripts/build_linux.sh --backend cuda --cuda-arch native --target audiocpp_cli --model-set custom --models yue2 --jobs $(nproc)

    for cand in (
        "/content/YuE/tools/audio.cpp/build/linux-cuda-release/bin/audiocpp_cli",
        "/content/YuE/tools/audio.cpp/build/bin/audiocpp_cli",
        "/content/YuE/tools/audio.cpp/build/audiocpp_cli",
        "/content/YuE/tools/audio.cpp/bin/audiocpp_cli"
    ):
        if os.path.isfile(cand):
            !cp {cand} /content/YuE/audiocpp_cli
            !cp {cand} /usr/local/bin/audiocpp_cli
            !chmod +x /content/YuE/audiocpp_cli /usr/local/bin/audiocpp_cli
            print(f"✅ Silnik audiocpp_cli (GGUF) pomyślnie zainstalowany z {cand}!")
            break

%cd /content/YuE

# Instalacja pakietu w środowisku Pythona (editable)
!pip install -q -e /content/YuE

# Tworzenie folderów na modele i wyniki
os.makedirs('/content/YuE/models/YuE2-3B', exist_ok=True)
os.makedirs('/content/YuE/models/YuE2-Vae', exist_ok=True)
os.makedirs('/content/YuE/models/Yue2-3B-GGUF', exist_ok=True)
os.makedirs('/content/YuE/models/loras', exist_ok=True)
os.makedirs('/content/YuE/runs/studio', exist_ok=True)

# Przygotowanie launchera Colab z tunelem Cloudflare i patchem autoryzacji
launch_code = '''"""Launch YuE2 Studio in Google Colab with GPU acceleration and Cloudflare Tunnel."""
from __future__ import annotations
import os, re, sys, time, site, secrets, threading, subprocess
from pathlib import Path

# Disable broken torchvision in transformers
os.environ['TRANSFORMERS_NO_TORCHVISION'] = '1'
if 'torchvision' in sys.modules: del sys.modules['torchvision']

src_dir = str(Path('/content/YuE/src').resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
os.environ['PYTHONPATH'] = f"{src_dir}:{os.environ.get('PYTHONPATH', '')}"

import yue2_studio.server as server_mod

# Patch security guard for public tunnel access (Cloudflare)
def colab_guard(self, write=False):
    if write and not secrets.compare_digest(self.headers.get('X-Studio-Token', ''), self.server.token):
        raise PermissionError('Studio session expired. Reload this page.')
server_mod.Handler.guard = colab_guard

# Allow port reuse and disable auto-stop on idle browser
server_mod.StudioServer.allow_reuse_address = True
server_mod.StudioServer.service_actions = lambda self: None

def start_cloudflare_tunnel(port: int = 7862):
    print("🌐 Inicjalizacja bezpiecznego tunelu Cloudflare...", flush=True)
    cloudflared_bin = "cloudflared"
    if subprocess.call(["which", "cloudflared"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) != 0:
        bin_path = Path("/tmp/cloudflared")
        if not bin_path.exists():
            subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", str(bin_path)], check=True)
            bin_path.chmod(0o755)
        cloudflared_bin = str(bin_path)
    cmd = [cloudflared_bin, "tunnel", "--url", f"http://127.0.0.1:{port}"]
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            print("\n" + "="*60, flush=True)
            print(f"🎉 TWÓJ LINK DO YUE2 STUDIO (Działa na telefonie & PC):", flush=True)
            print(f"👉 {url}", flush=True)
            print("="*60 + "\n", flush=True)
            break

def main():
    port = int(os.environ.get('PORT', 7862))
    runs_dir = Path('/content/YuE/runs/studio')
    runs_dir.mkdir(parents=True, exist_ok=True)
    threading.Thread(target=start_cloudflare_tunnel, args=(port,), daemon=True).start()
    print(f"🚀 Uruchamianie YuE2 Studio na porcie {port} (PyTorch BF16/CUDA)...", flush=True)
    server = server_mod.StudioServer(('0.0.0.0', port), root=runs_dir)
    try: server.serve_forever()
    except KeyboardInterrupt: pass
    finally:
        server.surprises.close()
        server.jobs.close()
        server.server_close()

if __name__ == '__main__': main()
'''
with open('/content/YuE/launch_colab.py', 'w', encoding='utf-8') as f:
    f.write(launch_code)

import torch
print("\n" + "="*50)
if torch.cuda.is_available():
    print(f"✅ Gotowe! Aktywne GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB VRAM)")
    print("✅ Moduły yue2, yue2_studio oraz silnik audiocpp_cli gotowe!")
print("="*50)


## 💾 Krok 1b (Opcjonalnie): Zapisz silnik `audiocpp_cli` (by uniknąć ponownej kompilacji w przyszłości)
Zapisuje skompilowany plik binarny `audiocpp_cli` (CUDA sm_75+) bezpośrednio na Twój **Dysk Google** lub pobiera go na komputer. Dzięki temu przy kolejnych sesjach Colaba instalacja trwa **1 sekundę zamiast 15 minut**!

In [ ]:
#@title 💾 Zapisz silnik audiocpp_cli (Google Drive / Pobierz na PC) 📦
zapisz_na_dysk_google = True #@param {type:"boolean"}
pobierz_na_komputer = False #@param {type:"boolean"}

if zapisz_na_dysk_google:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp /content/YuE/audiocpp_cli /content/drive/MyDrive/audiocpp_cli
    print("\n✅ Plik audiocpp_cli został pomyślnie zapisany na Twoim Dysku Google (/content/drive/MyDrive/audiocpp_cli)!")
    print("👉 Przy kolejnym uruchomieniu notebooka, Krok 1 automatycznie go załaduje w 1 sekundę!")

if pobierz_na_komputer:
    from google.colab import files
    print("📥 Pobieranie pliku audiocpp_cli na Twój komputer...")
    files.download('/content/YuE/audiocpp_cli')


## 📥 Krok 2: Pobranie Modeli YuE2 (3B + VAE + GGUF Q4 + LoRA)

In [ ]:
#@title 2. Pobieranie Modeli YuE2 📥
pobierz_pytorch_3b = False #@param {type:"boolean"}
pobierz_gguf_q4 = True #@param {type:"boolean"}
pobierz_instrumental_lora = True #@param {type:"boolean"}

%cd /content/YuE

if pobierz_pytorch_3b:
    print("📥 1. Pobieranie modelu głównego: m-a-p/YuE2-3B...")
    !huggingface-cli download m-a-p/YuE2-3B --local-dir models/YuE2-3B
    print("\n📥 2. Pobieranie dekodera VAE: m-a-p/YuE2-Vae...")
    !huggingface-cli download m-a-p/YuE2-Vae --local-dir models/YuE2-Vae

if pobierz_gguf_q4:
    print("\n📥 3. Pobieranie modelu GGUF Q4 (audio-cpp/Yue2-3B-GGUF)...")
    !huggingface-cli download audio-cpp/Yue2-3B-GGUF yue2-3b-q4_0.gguf --local-dir models/Yue2-3B-GGUF
    !huggingface-cli download audio-cpp/Yue2-3B-GGUF yue2-vae-f16.gguf --local-dir models/Yue2-3B-GGUF
    !huggingface-cli download audio-cpp/Yue2-3B-GGUF --include "sidecars/*" --local-dir models/Yue2-3B-GGUF

if pobierz_instrumental_lora:
    print("\n📥 4. Pobieranie Instrumental LoRA...")
    !huggingface-cli download Mothersuperior/YuE2-instrumental-cot-full-loras --local-dir models/loras/YuE2-instrumental-cot

print("\n✅ Modele pobrane i gotowe!")
print("💡 Wskazówka: W Studio WebUI w menu 'Music models' kliknij 'Use Q4' aby ustawić model Q4 jako aktywny silnik!")

## ✍️ Krok 2b (Opcjonalnie): Lokalny Asystent AI do Pisania Tekstów (Qwen 2.5 3B)
Uruchamia lokalnego asystenta songwritera na GPU przez Ollama. Pozwala automatycznie pisać teksty, zwrotki `[verse]`, refreny `[chorus]` oraz prowadzić naturalną rozmowę z **AI Music Producer Chat** bez żadnych zewnętrznych kluczy API!

In [ ]:
#@title ✍️ Opcjonalnie: Uruchom Asystenta AI (Qwen 2.5 3B) 🤖
uruchom_qwen_ai = True #@param {type:"boolean"}
model_qwen = "qwen2.5:3b" #@param ["qwen2.5:3b", "qwen2.5:1.5b", "qwen2.5:0.5b", "llama3.2:3b"]

if uruchom_qwen_ai:
    print(f"🤖 1. Instalacja silnika Ollama w Colabie...")
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
    import subprocess, time
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)
    print(f"📥 2. Pobieranie modelu {model_qwen} (ok. 1.9 GB na GPU)...")
    !ollama pull {model_qwen}
    print(f"\n✅ Lokalny Asystent AI ({model_qwen}) działa w tle na http://127.0.0.1:11434!")
    print(f"👉 W Studio WebUI: kliknij ikonę iskry (Songwriter) -> Wybierz Provider: Ollama -> Model: {model_qwen}")


## 🌐 Krok 3: Start YuE2 Studio WebUI

In [ ]:
#@title 3. Uruchom Pełne Studio YuE2 🎛️
%cd /content/YuE

# Zwolnienie zajętego portu 7862 jeśli poprzednia sesja wciąż działa w tle
!fuser -k 7862/tcp > /dev/null 2>&1 || true
!pkill -9 -f launch_colab.py > /dev/null 2>&1 || true

# Start Studio na GPU z linkiem Cloudflare
!python -u launch_colab.py
